# Steganography with Pillow

In this notebook, we will hide a secret message inside an image by modifying
the least significant bits of pixel values.


You will:
- Convert text to bits
- Modify pixel values
- Encode a message into an image
- Decode the message

In [ ]:
from PIL import Image

In [ ]:
# Load Image
# Make sure `thinker.png` is in the same directory.
img = Image.open("thinker.png")
pixels = img.load()
img

Use the `pixels` command to see the (R,G,B) values of a pixel - 

In [ ]:
pixels[0,0]

## Step 1: Convert Text to Bits

Each character has an ASCII value that can be found with Python's `ord` command.

In [ ]:
ord('Y')

This ASCII value can be converted to an 8-bit binary string using Python's `bin` command.  Note to state a number as binary, preface it with 0b.

In [ ]:
bin(89)

In [ ]:
0b001

Instead of using `bin` we will format the bit string in an f-string use the binary format "b".

In [ ]:
def text_to_bits(text):
    bits = ""
    for char in text:
        bits += f'{ord(char):08b}'
    return bits
text_to_bits("Hi")

## Step 2: Modify One Pixel

We will change the least significant bit of the red color of each pixel.  
First we will write a simple function to clear the last bit.

In [ ]:
# r // 2 floor divides, doubling it returns it except as even if it was odd
def clear_last_bit(num):
    return  num // 2 * 2

print(bin(clear_last_bit(0b1110111)))  # '0b1110110'

Now write a simple function to change the last bit to "new_bit".

In [ ]:
def change_last_bit(num, new_bit):
    return clear_last_bit(num) + new_bit

bin(change_last_bit(0b11011,0))

## Step 3: Encode a Message

We will loop through pixels and store one bit per pixel.

In [ ]:
# stores message in lsb of red channel of pixels
def encode_message(image_path, message, output_path):
    img = Image.open(image_path)
    pixels = img.load()

    bits = text_to_bits(message)
    # print(message[:20],'->',bits[:20])
    width, height = img.size

    i = 0
    for y in range(height):
        for x in range(width):
            if i >= len(bits):    # done encoding the message, save image and get out
                img.save(output_path)
                return

            r, g, b = pixels[x, y]

            # store current bit of message in least significant bit of red channel
            new_bit = int(bits[i])
            r = change_last_bit(r, new_bit)
            
            pixels[x, y] = (r, g, b)
            i += 1
    img.save(output_path)   # message outlasted the image size

## Step 4: Try Encoding
Run the function with a short message.

In [ ]:
encode_message("thinker.png", "Hello Data Science Fans!", "encoded_hello.png")

Compare before and after images - use a little Jupyter trick to borrow some html

In [ ]:
from IPython.display import HTML, display
# Ensure images are saved to disk in the same directory as the notebook

display(HTML("<table><tr><td><img src='thinker.png'></td><td><img src='encoded_hello.png'></td></tr></table>"))


Let's put a whole book (or as much as will fit) as the secret message.

In [ ]:
infile = open('red_headed_league.txt','r')
msg = infile.read()
infile.close()

In [ ]:
encode_message("thinker.png", msg, "encoded_rhl.png")

## Step 5: Decode the Message
Extract bits and rebuild the original text.

In [ ]:
def get_bits(image_path, length):
    img = Image.open(image_path)
    pixels = img.load()
    bits = ""
    width, height = img.size
    counter = 1
    for y in range(height):
        for x in range(width):
            r, g, b = pixels[x, y]
            lsb = r % 2         # get the lsb by dividing by 2 - even it's 0, odd it's 1
            bits += str(lsb)
            counter += 1
            if counter > length*8: return bits
            
def decode_message(image_path, length):
    bits = get_bits(image_path, length)
    #print('bits=',bits)
    chars = []
    for i in range(0, len(bits), 8):
        byte = bits[i:i+8]
        char = chr(int(byte, 2))   # chr converts byte string to ASCII character
        chars.append(char)
    msg = ''.join(chars)
    return msg

## Step 6: Try Decoding

In [ ]:
decode_message("encoded_hello.png", 30)

In [ ]:
decode_message("encoded_rhl.png", 930)

# Follow-on Questions

- Why does changing the last bit not noticeably change the image?
- How many characters can this image store?
- How would the code change if we wanted to use red and blue channels?  least two significant bits?